In [30]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())

['url', 'label']


In [32]:
import pandas as pd
import numpy as np
import re
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from sklearn.model_selection import train_test_split

In [33]:
df = pd.read_csv('urldata.csv')
df.columns = df.columns.str.strip()
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print(df.head())

Columns: ['Unnamed: 0', 'url', 'label', 'result']
Shape: (174434, 4)
   Unnamed: 0                        url   label  result
0           0     https://www.google.com  benign       0
1           1    https://www.youtube.com  benign       0
2           2   https://www.facebook.com  benign       0
3           3      https://www.baidu.com  benign       0
4           4  https://www.wikipedia.org  benign       0


In [34]:
df = df[['url', 'result']].dropna()
df.columns = ['url', 'label']
df['label'] = df['label'].astype(int)
print(f"Total    : {len(df)}")
print(f"Benign   : {(df['label']==0).sum()}")
print(f"Malicious: {(df['label']==1).sum()}")
print(df.head())

Total    : 174434
Benign   : 149998
Malicious: 24436
                         url  label
0     https://www.google.com      0
1    https://www.youtube.com      0
2   https://www.facebook.com      0
3      https://www.baidu.com      0
4  https://www.wikipedia.org      0


In [35]:
def extract_features(url):
    url = str(url).lower().strip()
    features = [
        len(url),
        url.count('.'),
        url.count('-'),
        url.count('/'),
        sum(c.isdigit() for c in url),
        len(re.findall(r'[^a-z0-9.\-/:]', url)),
        1 if url.startswith('https') else 0,
        len(url.split('/')[2]) if '//' in url else len(url),
        1 if any(t in url for t in ['.tk','.xyz','.ru','.pw']) else 0,
        len(url.split('/')[-1]),
    ]
    return features

print("Extracting features...")
X = np.array([extract_features(u) for u in df['url']], dtype=np.float32)
y = df['label'].values.astype(np.float32)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

Extracting features...
X shape: (174434, 10)
y shape: (174434,)


In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

Train: (139547, 10)  |  Test: (34887, 10)


In [38]:
model = Sequential([
    Input(shape=(10,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1,  activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                      │ (None, 64)                  │             704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,817 (11.00 KB)

 Trainable params: 2,817 (11.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
2181/2181 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.9482 - loss: 0.1712 - val_accuracy: 0.9683 - val_loss: 0.1115
Epoch 2/10
2181/2181 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9686 - loss: 0.1148 - val_accuracy: 0.9713 - val_loss: 0.1005
Epoch 3/10
2181/2181 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.9701 - loss: 0.1050 - val_accuracy: 0.9715 - val_loss: 0.0956
Epoch 4/10
2181/2181 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9707 - loss: 0.1001 - val_accuracy: 0.9718 - val_loss: 0.0899
Epoch 5/10
2181/2181 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9718 - loss: 0.0948 - val_accuracy: 0.9717 - val_loss: 0.0857
Epoch 6/10
2181/2181 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9733 - loss: 0.0901 - val_accuracy: 0.9757 - val_loss: 0.0798
Epoch 7/10
2181/2181 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9756 - loss: 0.0869 - val_accuracy: 0.9762 - val_loss: 0.0815
Epoch 8/10
2181/2181 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9777 - loss: 0.0821 

In [39]:
loss, acc = model.evaluate(X_test, y_test)
print(f"\n✅ Test Accuracy: {acc*100:.2f}%")

1091/1091 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9785 - loss: 0.0780

✅ Test Accuracy: 97.85%


In [40]:
model.save('url_spam_detector.h5')
print("✅ Saved as url_spam_detector.h5")

✅ Saved as url_spam_detector.h5


In [42]:
test_urls = [
    "https://www.google.com",
    "http://free-money.tk/click/win",
    "https://www.amazon.com/products/item",
    "http://192.168.1.1/phishing/login.php",
    "https://www.instagram.com"
]

print("\n--- Manual Test Results ---")
for url in test_urls:
    features   = np.array([extract_features(url)], dtype=np.float32)
    prediction = model.predict(features, verbose=0)[0][0]
    is_spam    = prediction >= 0.5
    label      = "🚨 SPAM/PHISHING" if is_spam else "✅ SAFE"
    confidence = prediction if is_spam else 1 - prediction
    print(f"{label} ({confidence*100:.1f}%) — {url}")


--- Manual Test Results ---
✅ SAFE (97.0%) — https://www.google.com
🚨 SPAM/PHISHING (100.0%) — http://free-money.tk/click/win
✅ SAFE (92.5%) — https://www.amazon.com/products/item
🚨 SPAM/PHISHING (99.8%) — http://192.168.1.1/phishing/login.php
✅ SAFE (96.3%) — https://www.instagram.com
